<a href="https://colab.research.google.com/github/FurkanAlpGurakan/AYM-Karar-Tahmini-NLP/blob/main/docx_to_excel_ipyn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install python-docx pandas openpyxl
# 1. ADIM: Gerekli Kütüphaneyi Kur (Colab'de python-docx yüklü gelmez)
!pip install python-docx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 5.6 MB/s eta 0:00:00


Word Belgelerinden Eğitim Veri Seti Oluşturulması

Bu script, Google Drive üzerinde sınıflara göre ayrılmış Word (.docx) formatındaki Anayasa Mahkemesi kararlarını işleyerek makine öğrenmesi için kullanılacak bir eğitim veri seti oluşturur.
Karar metinlerinden sonucu doğrudan ele veren bölümler ve tekrar eden başlıklar temizlenir; böylece modelin yalnızca karar gerekçesini öğrenmesi sağlanır.
Her alt klasör bir sınıf etiketi olarak kabul edilir, yeterli uzunluğa sahip metinler seçilerek etiketleriyle birlikte Excel formatında kaydedilir ve işlem sonunda veri setinin doğruluğu örnek bir kontrol ile doğrulanır.

In [ ]:
import os
import pandas as pd
from docx import Document
from google.colab import drive


if not os.path.exists('/content/drive'):
    print("Google Drive bağlanıyor...")
    drive.mount('/content/drive')

ANA_KLASOR = "/content/drive/MyDrive/Dava_Sonuc_Tahmin/Test_Verileri"
KAYIT_YOLU = "/content/drive/MyDrive/Dava_Sonuc_Tahmin/Test_Veriseti_Egitim.xlsx"


def metni_temizle_ve_kes(metin: str) -> str:
    """
    Metnin sonucunu ele veren kısımlarını (spoiler) keser.
    Başlıktaki 'KARAR' vs. kelimelerine dokunmamak için ilk 500 karaktere dokunmaz.
    Ayrıca bazı tekrar eden başlıkları (boilerplate) metinden temizler.
    """
    # 1) Boş yere yer kaplayan, her kararda tekrar eden bazı sabit başlıkları temizle
    boilerplate = [
        "İKİNCİ BÖLÜM KARAR",
        "I. BAŞVURUNUN ÖZETİ",
        "II. BAŞVURUNUN ÖZETİ",
        "I. BAŞVURU SÜRECİ",
        "II. BAŞVURU SÜRECİ",
        "BAŞVURUNUN ÖZETİ",
        "BAŞVURU SÜRECİ"
    ]
    for b in boilerplate:
        metin = metin.replace(b, " ")

    # 2) Sonuç kısmını kesmek için tetikleyiciler
    kesme_noktalari = [
        "Açıklanan gerekçelerle",  # En sık görülen sonuç başlangıcı
        "Bu gerekçelerle",
        "HÜKÜM",
        "SONUÇ",
        "YARGILAMA GİDERLERİ",
        "KARAR"  # Dikkat: Başlıkta da olabilir, bu yüzden indeks > 500 şartı var
    ]

    en_iyi_indeks = len(metin)
    kesen_kelime = None

    for nokta in kesme_noktalari:
        indeks = metin.find(nokta)

        # Baş tarafı (ilk 500 karakter) kesme, genelde başlıklar orada
        if indeks != -1 and indeks > 500:
            if indeks < en_iyi_indeks:
                en_iyi_indeks = indeks
                kesen_kelime = nokta

    if kesen_kelime:
        return metin[:en_iyi_indeks].strip()
    else:
        return metin.strip()

def wordleri_isle():
    # Klasör kontrolü
    if not os.path.exists(ANA_KLASOR):
        print(f" HATA: '{ANA_KLASOR}' yolu bulunamadı!")
        return

    print(f" FİNAL VERİ HAZIRLAYICI ÇALIŞIYOR...\nHedef klasör: {ANA_KLASOR}")
    veri_listesi = []

    # Ana klasör altındaki her klasör bir etiket (sınıf) olarak alınır
    for etiket in os.listdir(ANA_KLASOR):
        klasor_yolu = os.path.join(ANA_KLASOR, etiket)

        if os.path.isdir(klasor_yolu):
            print(f"\n Kategori İşleniyor: {etiket}")
            dosyalar = [f for f in os.listdir(klasor_yolu) if f.endswith('.docx')]

            sayac = 0
            for dosya_adi in dosyalar:
                dosya_yolu = os.path.join(klasor_yolu, dosya_adi)

                try:
                    doc = Document(dosya_yolu)
                    paragraflar = [p.text for p in doc.paragraphs if p.text.strip() != ""]
                    ham_metin = " ".join(paragraflar)

                    # Sonuç kısmını kes
                    temiz_metin = metni_temizle_ve_kes(ham_metin)

                    # Çok kısa metinleri alma
                    if len(temiz_metin) > 200:
                        veri_listesi.append({
                            "Dosya_Adi": dosya_adi,
                            "Metin": temiz_metin,
                            "Etiket": etiket
                        })
                        sayac += 1
                        if sayac % 20 == 0:
                            print(f"    {sayac}. dosya işlendi...")
                    else:
                        # Çok kısa olanlar loglanabilir istersen
                        pass

                except Exception as e:
                    print(f"  Hata ({dosya_adi}): {e}")

    # 3. Excel'e Kaydet
    if veri_listesi:
        df = pd.DataFrame(veri_listesi)
        df.to_excel(KAYIT_YOLU, index=False)

        print("\n" + "="*50)
        print(f" İŞLEM TAMAMLANDI!")
        print(f" Toplam İşlenen Karar Sayısı: {len(df)}")
        print(f" Dosya Kaydedildi: {KAYIT_YOLU}")
        print("="*50)

        # Kontrol için rastgele bir örnek göster
        print("\n ÖRNEK KONTROL (Son 100 Karakter):")
        ornek = df.sample(1).iloc[0]
        print(f"Etiket: {ornek['Etiket']}")
        print(f"Metin Sonu: ...{ornek['Metin'][-100:]}")
        print("(Burada 'ihlal edilmiştir' gibi doğrudan sonuç cümleleri olmaması gerekir.)")

    else:
        print("\n Hiç veri oluşturulamadı.")

# Script ana girişi
if __name__ == "__main__":
    wordleri_isle()


Google Drive bağlanıyor...
Mounted at /content/drive
 FİNAL VERİ HAZIRLAYICI ÇALIŞIYOR...
Hedef klasör: /content/drive/MyDrive/Dava_Sonuc_Tahmin/Test_Verileri

 Kategori İşleniyor: İhlal_Var
    20. dosya işlendi...
    40. dosya işlendi...

 Kategori İşleniyor: İhlal_Yok
    20. dosya işlendi...
    40. dosya işlendi...

 Kategori İşleniyor: Kabul_Edilemez
    20. dosya işlendi...
